# 04 – Combine niche model with SeaBee remote sensing

Combines the **niche model** (03b) with **SeaBee remote sensing** (shallow water)
into one dataset, which is the input to 05.

- **Niche model**: `nkm-utbredelsesmodell_norge_2026_25833.geo.parquet` – rule-based
  classification from light/salinity/substrate/intertidal.
- **SeaBee remote sensing**:
  `marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg`
  – the national regelC habitat product: observed habitat classes
  (ALGAE, ROCK, SEDIMENT, ANGIO, DEEP, ANTHRO) in shallow water.

The rules for how remote sensing overrides the niche model live in
`mnk.rules.classify_seabee` (one rule set, derived from the data model
`nkm_klassifiseringsregler`). ANGIO/ALGAE/ROCK/SEDIMENT are authoritative for
both substrate and light (`nkm_lys`).

**Performance**: the niche model is national (~2.7 M polygons). Only polygons
that actually overlap SeaBee habitat are touched – the rest pass through
unchanged without expensive geometry operations.

In [ ]:
import gc
import io
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.vectorize
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

NICHE_PARQUET = "gs://niva-geodata/MarintNaturKart/results/nkm-utbredelsesmodell_norge_2026_25833.geo.parquet"
SEABEE_GPKG = REPO / "marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg"

TORRFALL_PARQUET = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"

# Habitat SeaBee classes that can override the niche model.
# DEEP/ANTHRO do not override (see mnk.rules.SEABEE_RULES).
HABITAT_CLASSES = ["ALGAE", "ROCK", "SEDIMENT", "ANGIO"]

# Codes in the niche model (nkm_*) -> tokens in mnk.rules.
def _invert(d):
    return {v: k for k, v in d.items()}

LIGHT_MAP = _invert(mnk.rules.NKM_LYS)
SALINITY_MAP = _invert(mnk.rules.NKM_SALINITET)
SUBSTRATE_MAP = _invert(mnk.rules.NKM_SUBSTRAT)
INTERTIDAL_MAP = _invert(mnk.rules.NKM_TOERFALL)

## 1. Load data

In [ ]:
niche = gpd.read_parquet(NICHE_PARQUET)
assert niche.crs.to_epsg() == 25833
print(f"Niche model: {len(niche):,} polygons")

seabee = gpd.read_file(
    SEABEE_GPKG, layer="polygons", columns=["class_1", "usikkerhet_klasse"]
).to_crs(CRS)
seabee = seabee.rename(columns={"class_1": "klasse"})
seabee["_usikkerhet"] = seabee["usikkerhet_klasse"].replace({"middels": "medium"})
seabee_hab = seabee.loc[
    seabee["klasse"].isin(HABITAT_CLASSES), ["klasse", "_usikkerhet", "geometry"]
].reset_index(drop=True)
print(f"SeaBee habitat: {len(seabee_hab):,} polygons")
print(seabee["klasse"].value_counts())
print(seabee_hab["_usikkerhet"].value_counts())

del seabee
gc.collect()

## 2. Find niche model polygons overlapping SeaBee habitat

Spatial index query `sindex.query(..., predicate="intersects")` separates
**affected** polygons from those passed through unchanged. Only affected
polygons go on to the expensive intersection/difference.

In [ ]:
# Spatial index query (no global union_all – scales to the whole country).
hit = niche.sindex.query(seabee_hab.geometry, predicate="intersects")
hit_idx = niche.index[np.unique(hit[1])]

affected = niche.loc[hit_idx].copy()
unaffected = niche.drop(index=hit_idx)

print(f"Affected by SeaBee habitat: {len(affected):,}  (unchanged: {len(unaffected):,})")

del niche, hit, hit_idx
gc.collect()

## 3. Overlay: SeaBee habitat × affected niche polygons

`intersect_layers` builds the override pieces (affected ∩ habitat) and
`subtract_layer` the remainder that stays niche model (affected − habitat). Both
resolve only the overlapping pairs through an STRtree bulk query; the per-polygon
difference in `subtract_layer` runs across threads.

In [ ]:
affected = affected.assign(
    _lys=affected[mnk.rules.COL_LYS].map(LIGHT_MAP),
    _sal=affected[mnk.rules.COL_SALINITET].map(SALINITY_MAP),
    _sub=affected[mnk.rules.COL_SUBSTRAT].map(SUBSTRATE_MAP),
    _tf=affected[mnk.rules.COL_TOERFALL].map(INTERTIDAL_MAP),
)

overlaid = mnk.vectorize.intersect_layers(
    affected, seabee_hab, workers=4, progress=True
).explode(index_parts=False)
overlaid = overlaid[~overlaid.geometry.is_empty & overlaid.geometry.notna()]

affected_rest = mnk.vectorize.subtract_layer(affected, seabee_hab, workers=4, progress=True)
# SeaBee habitat area not covered by any nisjemodell polygon (whole polygons
# and the outside parts of partly overlapping ones); classified in §7.
seabee_only = mnk.vectorize.subtract_layer(seabee_hab, affected, workers=4, progress=True)
seabee_only = seabee_only.explode(index_parts=False).reset_index(drop=True)
seabee_only = seabee_only[~seabee_only.geometry.is_empty & seabee_only.geometry.notna()]
print(f"SeaBee-only (no nisjemodell): {len(seabee_only):,}")
print(f"Overlay pieces: {len(overlaid):,}  | affected remainder: {len(affected_rest):,}")

del affected, seabee_hab
gc.collect()

## 4. Apply `classify_seabee` (vectorised over unique combinations)

The number of unique `(klasse, lys, salinitet, substrat, tørrfall)` combinations
is small, so the rules are resolved once per combination and scattered back to
every piece by integer code (`factorize` + gather) – no per-polygon Python loop
and no 5-key object merge over the geometry-bearing frame.

In [ ]:
keys = ["klasse", "_lys", "_sal", "_sub", "_tf"]

kcols = [overlaid[k].astype("string").fillna("\x00") for k in keys]
code = pd.factorize(kcols[0].str.cat(kcols[1:], sep="\x01"), sort=False)[0]
first = np.unique(code, return_index=True)[1]
combos = overlaid[keys].iloc[first].reset_index(drop=True)
del kcols

nin_code = np.empty(len(combos), dtype=object)
nin_name = np.empty(len(combos), dtype=object)
for i, combo in enumerate(combos.itertuples(index=False, name=None)):
    res = mnk.rules.classify_seabee(*combo)
    nin_code[i], nin_name[i] = res if res is not None else (pd.NA, pd.NA)
    print(f"\r  classify_seabee: {i + 1:,}/{len(combos):,} combos", end="", flush=True)
print()

combos["_nin_code"], combos["_nin_name"] = nin_code, nin_name
print(combos.to_string())

overlaid["_nin_code"] = nin_code[code]
overlaid["_nin_name"] = nin_name[code]
overlaid = overlaid[overlaid["_nin_code"].notna()].copy()

## 5. Update attributes of the remotely sensed pieces

- `nin_kode_fkid` / `NiN_name`: from the remote sensing rule.
- `nkm_prosess`: looked up from `mnk.attributes.NKM_PROSESS`.
- `nkm_lm-dk`: set to the remotely sensed substrate.
- `nkm_lys`: set from `mnk.rules.SEABEE_LYS_OVERRIDE` (bottom observed ⇒ eufotisk).
- `nkm_sa-fa`: fastsittende-alger basistrinn from the SeaBee class — ALGAE ⇒ `ABC`,
  ROCK ⇒ `0B`; other classes leave it NA.
- `nkm_usikkerhetsklasse`: read directly from the SeaBee `usikkerhet_klasse` field
  (`middels` renamed to `medium`); `metode_usikkerhet` is `NKM_satelitt_habitat`.
- `datakilde`: SeaBee is authoritative for lys + substrat (`lys_fjernmaalt_seabee`,
  `substrat_fjernmaalt_seabee`); salinitet stays HI and tørrfall stays kartverket.
- `metode_klassifisering`: the `|`-joined k_metode codes derived from `datakilde`
  via `mnk.attributes.metode_from_datakilde` (SeaBee ⇒ `NKM_modell_satellitt`,
  plus `NKM_modell_salinitet` and, in the intertidal, `NKM_regelbasert_fjaeresone`).

In [ ]:
# Remote sensing overrides the model substrate. ANGIO (undervannseng) is soft bottom.
SEABEE_SUBSTRATE_NKM = {
    "ALGAE": mnk.rules.NKM_SUBSTRAT["fast"],
    "ROCK": mnk.rules.NKM_SUBSTRAT["fast"],
    "SEDIMENT": mnk.rules.NKM_SUBSTRAT["løst"],
    "ANGIO": mnk.rules.NKM_SUBSTRAT["løst"],
}
SEABEE_LYS_NKM = {k: mnk.rules.NKM_LYS[v] for k, v in mnk.rules.SEABEE_LYS_OVERRIDE.items()}
SEABEE_SA_FA_NKM = {k: mnk.rules.NKM_SA_FA[v] for k, v in mnk.rules.SEABEE_SA_FA_OVERRIDE.items()}

overlaid["nin_kode_fkid"] = overlaid["_nin_code"]
overlaid["NiN_name"] = overlaid["_nin_name"]
overlaid["nkm_prosess"] = overlaid["nin_kode_fkid"].map(mnk.attributes.NKM_PROSESS)
overlaid[mnk.rules.COL_SUBSTRAT] = (
    overlaid["klasse"].map(SEABEE_SUBSTRATE_NKM).fillna(overlaid[mnk.rules.COL_SUBSTRAT])
)
overlaid[mnk.rules.COL_LYS] = (
    overlaid["klasse"].map(SEABEE_LYS_NKM).fillna(overlaid[mnk.rules.COL_LYS])
)
overlaid[mnk.rules.COL_SA_FA] = overlaid["klasse"].map(SEABEE_SA_FA_NKM).astype("string")

overlaid["nkm_usikkerhetsklasse"] = overlaid["_usikkerhet"]
overlaid["metode_usikkerhet"] = mnk.attributes.METODE_USIKKERHET_HABITAT

overlaid["datakilde"] = mnk.attributes.build_datakilde(
    np.full(len(overlaid), mnk.attributes.SOURCE_SUBSTRAT_SEABEE),
    overlaid[mnk.rules.COL_TOERFALL],
    lys_token=mnk.attributes.SOURCE_LYS_SEABEE,
)
overlaid["metode_klassifisering"] = mnk.attributes.metode_from_datakilde(overlaid["datakilde"])
assert overlaid["nkm_prosess"].notna().all()

## 6. Combine

`combined = unchanged niche model + affected remainder + remotely sensed pieces`.
Only a small minority of polygons went through geometry operations.

`nkm_seabee_klasse` keeps the SeaBee observed class (ALGAE/ROCK/SEDIMENT/ANGIO) for
the remotely sensed pieces, and is NA for the pure nisjemodell pieces.

`metode_klassifisering` is (re)derived here for every classified row from its
`datakilde` via `mnk.attributes.metode_from_datakilde` — the `|`-joined k_metode
codes of the contributing sources (e.g. `NKM_modell_lyssone|NKM_modell_salinitet|
NKM_modell_bunntype`). The SeaBee-only pieces added in §7 are stamped the same way.

In [ ]:
unaffected["nkm_seabee_klasse"] = pd.NA
affected_rest["nkm_seabee_klasse"] = pd.NA
overlaid["nkm_seabee_klasse"] = overlaid["klasse"]
unaffected[mnk.rules.COL_SA_FA] = pd.NA
affected_rest[mnk.rules.COL_SA_FA] = pd.NA

OUT_COLS = [c for c in unaffected.columns if not c.startswith("_") and c not in ("nin_int", "nk_fid")]

stamped = gpd.GeoDataFrame(
    pd.concat(
        [unaffected[OUT_COLS], affected_rest[OUT_COLS], overlaid[OUT_COLS]],
        ignore_index=True,
    ),
    crs=CRS,
)

classified = stamped["nin_kode_fkid"].notna()
stamped.loc[classified, "metode_klassifisering"] = mnk.attributes.metode_from_datakilde(
    stamped.loc[classified, "datakilde"]
)

print(f"Stamped (niche ⊕ SeaBee): {len(stamped):,} polygons")
print(stamped["nin_kode_fkid"].value_counts())
print(stamped["metode_klassifisering"].value_counts())

del unaffected, affected_rest, overlaid
gc.collect()

## 7. Add SeaBee habitat with no nisjemodell underneath

The nisjemodell does not reach the whole shallow zone (it clips tørrfall against
its own polygons, and intertidal areas with missing salinity get no polygon), so
`seabee_only` — the SeaBee habitat area left after subtracting the nisjemodell polygons in §3 — is
classified here directly from remote sensing and appended.

For a habitat class the rules need only **salinitet** and **tørrfall** (`nkm_lys`
is eufotisk and the substrate comes from the SeaBee class):

- **salinitet**: `brakk` if the SeaBee polygon touches a nisjemodell polygon with
  brakk salinitet (`nkm_lm-sa`), else `salt`.
- **tørrfall**: each polygon is split at the kartverket tørrfall edge
  (`TORRFALL_PARQUET`); the part inside is `innenfor`, the rest `utenfor`.

Attributes are stamped the same way as the overlaid pieces in §5, and `nkm_lm-sa` /
`nkm_lm-tv` are set from the derived salinitet / tørrfall. There is **no NA
tørrfall canvas** and **no deep-sea canvas**: areas covered by neither the
nisjemodell nor SeaBee stay an honest hole rather than a NA-filled or guessed
polygon.

In [ ]:
# Salinitet: brakk if the SeaBee polygon touches a brakk nisjemodell polygon, else salt.
brakk = stamped.geometry.values[
    (stamped[mnk.rules.COL_SALINITET] == mnk.rules.NKM_SALINITET["brakk"]).to_numpy()
]
is_brakk = np.zeros(len(seabee_only), dtype=bool)
b_hit = shapely.STRtree(brakk).query(seabee_only.geometry.values, predicate="intersects")
is_brakk[np.unique(b_hit[0])] = True
seabee_only["_sal"] = np.where(is_brakk, "brakk", "salt")
del brakk, b_hit

# Tørrfall: split each polygon at the tørrfall edge – inside part innenfor, rest utenfor.
torrfall = gpd.read_parquet(TORRFALL_PARQUET).to_crs(CRS)[["geometry"]].reset_index(drop=True)
torrfall["geometry"] = shapely.make_valid(torrfall.geometry.to_numpy())
tf_in = mnk.vectorize.intersect_layers(seabee_only, torrfall, workers=4).assign(_tf="innenfor")
tf_out = mnk.vectorize.subtract_layer(seabee_only, torrfall, workers=4).assign(_tf="utenfor")
del torrfall
gc.collect()

so = gpd.GeoDataFrame(pd.concat([tf_in, tf_out], ignore_index=True), crs=CRS)
so = so.explode(index_parts=False).reset_index(drop=True)
so = so[~so.geometry.is_empty & so.geometry.notna()].copy()
print(f"SeaBee-only pieces: {len(so):,} ({(so['_tf'] == 'innenfor').sum():,} innenfor tørrfall, "
      f"{(so['_sal'] == 'brakk').sum():,} brakk)")
del tf_in, tf_out

combos = so[["klasse", "_sal", "_tf"]].drop_duplicates().reset_index(drop=True)
res = [mnk.rules.classify_seabee(k, "eufotisk", s, "løst", t)
       for k, s, t in combos.itertuples(index=False, name=None)]
combos["nin_kode_fkid"] = [r[0] if r else pd.NA for r in res]
combos["NiN_name"] = [r[1] if r else pd.NA for r in res]
so = so.merge(combos, on=["klasse", "_sal", "_tf"], how="left")
so = so[so["nin_kode_fkid"].notna()].copy()

so["opphav_kode"] = mnk.attributes.OPPHAV_KODE
so["nkm_prosess"] = so["nin_kode_fkid"].map(mnk.attributes.NKM_PROSESS)
so[mnk.rules.COL_SUBSTRAT] = so["klasse"].map(SEABEE_SUBSTRATE_NKM)
so[mnk.rules.COL_LYS] = so["klasse"].map(SEABEE_LYS_NKM)
so[mnk.rules.COL_SALINITET] = so["_sal"].map(mnk.rules.NKM_SALINITET)
so[mnk.rules.COL_TOERFALL] = so["_tf"].map(mnk.rules.NKM_TOERFALL)
so[mnk.rules.COL_SA_FA] = so["klasse"].map(SEABEE_SA_FA_NKM).astype("string")
so["nkm_usikkerhetsklasse"] = so["_usikkerhet"]
so["metode_usikkerhet"] = mnk.attributes.METODE_USIKKERHET_HABITAT
so["nkm_seabee_klasse"] = so["klasse"]
so["datakilde"] = mnk.attributes.build_datakilde(
    np.full(len(so), mnk.attributes.SOURCE_SUBSTRAT_SEABEE),
    so[mnk.rules.COL_TOERFALL],
    lys_token=mnk.attributes.SOURCE_LYS_SEABEE,
)
so["metode_klassifisering"] = mnk.attributes.metode_from_datakilde(so["datakilde"])

final = gpd.GeoDataFrame(
    pd.concat([stamped, so[stamped.columns]], ignore_index=True), crs=CRS
)
print(f"Coverage: {len(final):,} polygons ({len(so):,} SeaBee-only classified)")
print(so["nin_kode_fkid"].value_counts())
print(final["nin_kode_fkid"].value_counts(dropna=False))

del stamped, seabee_only, so
gc.collect()

## 8. Save

In [ ]:
fname = mnk.utils.to_filename("nisjemodel-remote-sensing", "norge", "2026", CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"

obj_cols = final.select_dtypes("object").columns
final[obj_cols] = final[obj_cols].astype("string")

_meta = io.BytesIO()
final.iloc[:0].to_parquet(_meta)
_meta.seek(0)
geo_meta = pq.read_schema(pa.PythonFile(_meta, mode="r")).metadata[b"geo"]
schema = pa.table(final.iloc[:0].to_arrow(geometry_encoding="WKB")).schema.with_metadata(
    {b"geo": geo_meta}
)

CHUNK = 100_000
with pq.ParquetWriter(parquet_path, schema, compression="snappy") as w:
    for start in range(0, len(final), CHUNK):
        part = pa.table(final.iloc[start : start + CHUNK].to_arrow(geometry_encoding="WKB"))
        w.write_table(part.cast(schema))
        del part
        gc.collect()
print(f"Saved {len(final):,} polygons -> {parquet_path.name}")